# 02 Two-Telescope Interference

**Series:** Physics of the CHARA Array (Mount Wilson Observatory)
**Topic T2:** how light from two telescopes makes fringes, why the path lengths must match, and how precisely.

Notebook 01 ended with a picture: two mirrors $B$ apart produce fringes with spacing $\lambda/B$. This notebook looks at where those fringes come from and what it takes to see them.

1. Two waves add. Whether they reinforce or cancel depends on the path difference.
2. A star off to one side puts a large path difference between the telescopes.
3. Starlight contains many wavelengths, so fringes only appear when the paths match to within a few microns.
4. The **contrast** of the fringes carries information about the star.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

RAD_TO_MAS = 180 / np.pi * 3600 * 1000  # radians -> milliarcseconds
MAS_TO_RAD = 1 / RAD_TO_MAS

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8a85"
SLIDER_STYLE = {"handle_color": BLUE}  # visible knob on white backgrounds
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "lines.linewidth": 2,
})

## 1. Two waves add

Light from a star reaches telescope 1 and telescope 2. CHARA brings both beams to a lab and overlaps them in a **beam combiner**. What the detector measures depends on how the two waves line up when they meet.

- If one beam travelled exactly a whole number of wavelengths further, the crests line up and the light is **bright**.
- If it travelled half a wavelength further, crest meets trough and the light **cancels**.

The extra distance one beam travelled is the **optical path difference** (OPD). For two equal beams the detected intensity is

$$I = 1 + \cos\!\left(\frac{2\pi\,\text{OPD}}{\lambda}\right)$$

which swings between 0 and 2. The slider sets the OPD in units of wavelength.

In [2]:
@interact(
    opd_waves=FloatSlider(value=0.25, min=0.0, max=2.0, step=0.05,
                          description="OPD (λ)", continuous_update=False,
                          style=SLIDER_STYLE),
)
def show_two_waves(opd_waves):
    x = np.linspace(0, 3, 600)  # distance along the beam, in wavelengths
    wave1 = np.cos(2 * np.pi * x)
    wave2 = np.cos(2 * np.pi * (x - opd_waves))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(x, wave1, color=GREY, linewidth=1.2, linestyle="--", label="beam 1")
    ax1.plot(x, wave2, color=ORANGE, linewidth=1.2, linestyle="--", label="beam 2")
    ax1.plot(x, wave1 + wave2, color=BLUE, label="sum")
    ax1.set_ylim(-2.2, 3.2)
    ax1.set_xlabel("Position along the beam (wavelengths)")
    ax1.set_ylabel("Electric field")
    ax1.set_title(f"Waves offset by {opd_waves:.2f} λ")
    ax1.legend(fontsize=8, loc="upper right", ncol=3)

    opd = np.linspace(0, 2, 400)
    ax2.plot(opd, 1 + np.cos(2 * np.pi * opd), color=BLUE)
    ax2.scatter([opd_waves], [1 + np.cos(2 * np.pi * opd_waves)], s=80,
                color=ORANGE, zorder=3)
    ax2.set_xlabel("OPD (wavelengths)")
    ax2.set_ylabel("Detected intensity")
    ax2.set_title("Intensity vs path difference")
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.25, continuous_update=False, description='OPD (λ)', max=2.0, step=0.…

**Try this:** set OPD to 0.5 and the sum is flat, so the light cancels. At 1.0 it is bright again. The intensity curve on the right is a **fringe**: bright, dark, bright, repeating every wavelength of path difference.

## 2. Where the path difference comes from

A star is so far away that its light arrives as flat wavefronts. If the star is straight overhead, both telescopes catch the same wavefront at the same moment and the OPD is zero. If the star is off to one side, the wavefront reaches one telescope first. The other telescope's light must travel an extra

$$\text{OPD}_\text{geometric} = B \sin\theta$$

where $\theta$ is the star's angle from overhead, measured along the baseline. The diagram is drawn to scale.

In [3]:
@interact(
    baseline_m=FloatSlider(value=331, min=34, max=331, step=1,
                           description="B (m)", continuous_update=False,
                           style=SLIDER_STYLE),
    angle_deg=FloatSlider(value=30, min=-60, max=60, step=1,
                          description="θ (deg)", continuous_update=False,
                          style=SLIDER_STYLE),
)
def show_geometric_delay(baseline_m, angle_deg):
    theta = np.radians(angle_deg)
    t1 = np.array([-baseline_m / 2, 0.0])
    t2 = np.array([baseline_m / 2, 0.0])
    toward_star = np.array([np.sin(theta), np.cos(theta)])
    delay = baseline_m * np.sin(theta)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.axhline(0, color=GREY, linewidth=1)
    ax.scatter(*zip(t1, t2), s=120, marker="s", color=BLUE, zorder=3)
    ax.annotate("T1", t1, xytext=(0, -18), textcoords="offset points", ha="center")
    ax.annotate("T2", t2, xytext=(0, -18), textcoords="offset points", ha="center")

    # The wavefront that has just reached the telescope nearer the star
    near, far = (t2, t1) if delay >= 0 else (t1, t2)
    along_front = np.array([np.cos(theta), -np.sin(theta)])
    span = baseline_m * 0.9
    front = np.array([near - span * along_front, near + span * along_front])
    ax.plot(front[:, 0], front[:, 1], color=GREY, linewidth=1.2, linestyle="--",
            label="wavefront")

    # Starlight rays coming down to each telescope
    for t in (t1, t2):
        start = t + 1.2 * baseline_m * toward_star
        ax.annotate("", xy=t, xytext=start,
                    arrowprops=dict(arrowstyle="->", color=GREY, lw=1))

    # Extra path: from the wavefront down to the far telescope
    foot = far + abs(delay) * toward_star
    ax.plot([far[0], foot[0]], [far[1], foot[1]], color=ORANGE, linewidth=3,
            label=f"extra path = B sin θ = {abs(delay):.1f} m")

    ax.set_aspect("equal")
    ax.set_xlim(-1.1 * baseline_m, 1.1 * baseline_m)
    ax.set_ylim(-0.15 * baseline_m, 1.0 * baseline_m)
    ax.set_xlabel("m")
    ax.set_ylabel("m")
    ax.set_title(f"Star {angle_deg:.0f}° from overhead, baseline {baseline_m:.0f} m")
    ax.legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.show()

    print(f"Geometric OPD: {abs(delay):.2f} m")
    print(f"That is {abs(delay) / 1.6e-6:,.0f} wavelengths at 1.6 µm.")
    shift = baseline_m * MAS_TO_RAD
    print(f"Moving the star by 1 mas changes the OPD by {shift * 1e6:.2f} µm.")

interactive(children=(FloatSlider(value=331.0, continuous_update=False, description='B (m)', max=331.0, min=34…

Two numbers from that cell matter:

- **The OPD can be hundreds of metres.** CHARA must add the same distance to the other beam's path, or the beams never interfere. Its **delay lines** do that. Topic T6 covers how.
- **One milliarcsecond on the sky is about one wavelength of OPD** on a 331 m baseline. This is why fringes spaced $\lambda/B$ apart on the sky (notebook 01) correspond to one wavelength of path here. Tiny changes in star position become measurable shifts in the fringes.

## 3. Starlight has many colours: the fringe packet

A laser has one wavelength, so its fringe repeats forever as the OPD changes. Starlight passes through a filter that lets in a band of wavelengths. CHARA's H band, for example, runs from about 1.5 to 1.8 µm.

Each wavelength makes its own fringe with its own spacing. At OPD = 0 all of them are bright together. Further away they drift out of step and wash each other out. The result is a short **fringe packet**.

The packet's length is the **coherence length**

$$L_c \approx \frac{\lambda^2}{\Delta\lambda}$$

where $\Delta\lambda$ is the width of the band. The packet holds about $\lambda/\Delta\lambda$ fringes.

In [4]:
@interact(
    center_um=FloatSlider(value=1.65, min=0.5, max=2.2, step=0.05,
                          description="λ (µm)", continuous_update=False,
                          style=SLIDER_STYLE),
    bandwidth_um=FloatSlider(value=0.30, min=0.005, max=0.5, step=0.005,
                             description="Δλ (µm)", readout_format=".3f",
                             continuous_update=False, style=SLIDER_STYLE),
)
def show_fringe_packet(center_um, bandwidth_um):
    wavelengths = np.linspace(center_um - bandwidth_um / 2,
                              center_um + bandwidth_um / 2, 301)
    opd = np.linspace(-30, 30, 4001)  # µm
    fringes = 1 + np.cos(2 * np.pi * opd[None, :] / wavelengths[:, None])
    packet = fringes.mean(axis=0)
    coherence_um = center_um ** 2 / bandwidth_um

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
    for i, colour in zip([0, 150, 300], [ORANGE, AQUA, GREY]):
        ax1.plot(opd, fringes[i], color=colour, linewidth=1,
                 label=f"{wavelengths[i]:.3f} µm")
    ax1.set_ylabel("Intensity")
    ax1.set_title("Three of the wavelengths in the band, each on its own")
    ax1.legend(fontsize=8, loc="upper right")

    ax2.plot(opd, packet, color=BLUE)
    ax2.axvspan(-coherence_um / 2, coherence_um / 2, color=ORANGE, alpha=0.12,
                label=f"coherence length ≈ {coherence_um:.1f} µm")
    ax2.set_xlabel("OPD (µm)")
    ax2.set_ylabel("Intensity")
    ax2.set_title("All wavelengths together: the fringe packet")
    ax2.legend(fontsize=8, loc="upper right")
    ax2.set_xlim(opd[0], opd[-1])
    plt.tight_layout()
    plt.show()

    print(f"Coherence length λ²/Δλ: {coherence_um:.1f} µm")
    print(f"Fringes in the packet:  ~{center_um / bandwidth_um:.0f}")

interactive(children=(FloatSlider(value=1.65, continuous_update=False, description='λ (µm)', max=2.2, min=0.5,…

**Try this:**

- At the default H band (1.65 µm, 0.3 µm wide), the packet is about 9 µm long.
- Narrow Δλ to 0.01 µm, as a spectrograph does when it splits light into narrow channels. The packet stretches past the plot edges.

This is the hard engineering fact behind CHARA. The geometric OPD from section 2 can be **hundreds of metres**. The delay lines must cancel it to within a few **microns** to sit inside the packet, and keep tracking it as Earth rotates.

## 4. Fringe contrast carries the information

So far the source has been a single point. Fringes from a point swing all the way from 0 to full brightness. The depth of the swing is measured by the **visibility**

$$V = \frac{I_\text{max} - I_\text{min}}{I_\text{max} + I_\text{min}}$$

A point source gives $V = 1$.

Now make the source two stars close together, a binary. Each star makes its own fringe. From section 2, the second star sits at a slightly different angle, so its fringe is shifted by $B\,\Delta\theta$ of OPD. The detector sees the sum.

- If the shift is a whole number of wavelengths, the fringes line up and $V$ stays high.
- If the shift is half a wavelength, one star's bright fringe fills the other's dark fringe and $V$ drops.

The sliders use 1.6 µm light.

In [5]:
WAVELENGTH_UM = 1.6


def binary_visibility(baseline_m, separation_mas, flux_ratio):
    phase = 2 * np.pi * baseline_m * separation_mas * MAS_TO_RAD / (WAVELENGTH_UM * 1e-6)
    return np.abs(1 + flux_ratio * np.exp(1j * phase)) / (1 + flux_ratio)


@interact(
    baseline_m=FloatSlider(value=100, min=34, max=331, step=1,
                           description="B (m)", continuous_update=False,
                           style=SLIDER_STYLE),
    separation_mas=FloatSlider(value=2.0, min=0.0, max=5.0, step=0.05,
                               description="sep (mas)", continuous_update=False,
                               style=SLIDER_STYLE),
    flux_ratio=FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                           description="B/A flux", continuous_update=False,
                           style=SLIDER_STYLE),
)
def show_binary_fringes(baseline_m, separation_mas, flux_ratio):
    opd_um = np.linspace(-4 * WAVELENGTH_UM, 4 * WAVELENGTH_UM, 1200)
    shift_um = baseline_m * separation_mas * MAS_TO_RAD * 1e6
    star_a = 1 + np.cos(2 * np.pi * opd_um / WAVELENGTH_UM)
    star_b = flux_ratio * (1 + np.cos(2 * np.pi * (opd_um - shift_um) / WAVELENGTH_UM))
    total = (star_a + star_b) / (1 + flux_ratio)
    vis = binary_visibility(baseline_m, separation_mas, flux_ratio)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax1.plot(opd_um, star_a / (1 + flux_ratio), color=GREY, linewidth=1,
             linestyle="--", label="star A")
    ax1.plot(opd_um, star_b / (1 + flux_ratio), color=ORANGE, linewidth=1,
             linestyle="--", label="star B")
    ax1.plot(opd_um, total, color=BLUE, label="sum (what is detected)")
    ax1.set_ylim(0, 2.1)
    ax1.set_xlabel("OPD (µm)")
    ax1.set_ylabel("Intensity")
    ax1.set_title(f"Fringe shift between stars: {shift_um:.2f} µm, V = {vis:.2f}")
    ax1.legend(fontsize=8, loc="upper right")

    baselines = np.linspace(0, 331, 800)
    ax2.plot(baselines, binary_visibility(baselines, separation_mas, flux_ratio),
             color=BLUE)
    ax2.axvspan(0, 34, color=GREY, alpha=0.12, label="shorter than CHARA's 34 m")
    ax2.scatter([baseline_m], [vis], s=80, color=ORANGE, zorder=3)
    ax2.set_ylim(0, 1.05)
    ax2.set_xlabel("Baseline (m)")
    ax2.set_ylabel("Visibility V")
    ax2.set_title("Visibility vs baseline for this binary")
    ax2.legend(fontsize=8, loc="lower left")
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=100.0, continuous_update=False, description='B (m)', max=331.0, min=34…

**Try this:**

- Set the separation to 0. The two fringes coincide and $V = 1$, the same as a single star.
- With equal stars, find the baseline where $V$ hits 0. The fringes are exactly half a wavelength out of step.
- Lower the flux ratio. The dips get shallower because the faint star cannot fully fill the bright star's dark fringes.

The right-hand curve is the key result. **How visibility changes with baseline encodes the source's shape.** A binary gives a repeating pattern. A star's disk gives a different one (topic T4). Topic T3 shows the general rule behind this.

## Summary

| Idea | Formula | CHARA scale |
|---|---|---|
| Fringe intensity | $1 + \cos(2\pi\,\text{OPD}/\lambda)$ | one fringe per 1.6 µm of OPD in H band |
| Geometric delay | $B \sin\theta$ | up to hundreds of metres |
| Coherence length | $\lambda^2/\Delta\lambda$ | ~9 µm in H band |
| Sky angle to OPD | $B\,\Delta\theta$ | 1 mas ≈ 1.6 µm at 331 m |
| Visibility | $(I_\text{max}-I_\text{min})/(I_\text{max}+I_\text{min})$ | 1 for a point, lower for resolved sources |

1. Two beams interfere. Brightness depends on their path difference in wavelengths.
2. A star off overhead creates a path difference of up to hundreds of metres. Delay lines must cancel it.
3. Starlight's range of wavelengths limits fringes to a packet a few microns long. The paths must match to that precision.
4. Fringe contrast drops when the source has structure. Measuring contrast at many baselines reveals the source's shape.

**Next:** T3, the van Cittert–Zernike theorem, the general rule that links visibility to the image of the source.